In [1]:
import os
import pandas as pd
from pathlib import Path

# ============================================================
# 1. PROJECT PATH
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/PROJECT SAINS DATA")

DATA_DIR = BASE_DIR / "DATA"

DATASETS = {
    "Compound_Jamur": {
        "pubchem_compounds": DATA_DIR / "Compound_Jamur" / "pubchem_compounds.csv",
        "validated_compounds": DATA_DIR / "Compound_Jamur" / "validated_compounds.csv",
    },
    "MAO-B": {
        "target_metadata": DATA_DIR / "MAO-B" / "maob_target_metadata.csv",
        "activity_raw": DATA_DIR / "MAO-B" / "maob_activity_raw.csv",
    },
    "Alpha-Synuclein": {
        "target_metadata": DATA_DIR / "Alpha-Synuclein" / "alpha_synuclein_target_metadata.csv",
        "activity_raw": DATA_DIR / "Alpha-Synuclein" / "alpha_synuclein_activity_raw.csv",
        "assay_metadata": DATA_DIR / "Alpha-Synuclein" / "alpha_synuclein_assay_metadata_raw.csv",
    }
}

# ============================================================
# 2. FILE INVENTORY
# ============================================================

print("=" * 70)
print("DATASET INVENTORY")
print("=" * 70)

loaded_data = {}
inventory_results = []

for category, files in DATASETS.items():

    print(f"\n[{category}]")

    for dataset_name, file_path in files.items():

        if file_path.exists():

            try:
                df = pd.read_csv(file_path, low_memory=False)

                loaded_data[dataset_name] = df

                inventory_results.append({
                    "Category": category,
                    "Dataset": dataset_name,
                    "File_Exists": True,
                    "Rows": len(df),
                    "Columns": len(df.columns),
                    "File_Size_MB": round(file_path.stat().st_size / (1024**2), 3)
                })

                print(
                    f"{dataset_name}: FOUND | "
                    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
                )

            except Exception as e:

                inventory_results.append({
                    "Category": category,
                    "Dataset": dataset_name,
                    "File_Exists": True,
                    "Rows": None,
                    "Columns": None,
                    "File_Size_MB": None
                })

                print(f"{dataset_name}: ERROR READING FILE")
                print("Error:", e)

        else:

            inventory_results.append({
                "Category": category,
                "Dataset": dataset_name,
                "File_Exists": False,
                "Rows": None,
                "Columns": None,
                "File_Size_MB": None
            })

            print(f"{dataset_name}: NOT FOUND")

inventory_df = pd.DataFrame(inventory_results)

# ============================================================
# 3. BASIC DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("BASIC DATA QUALITY CHECK")
print("=" * 70)

quality_results = []

for dataset_name, df in loaded_data.items():

    missing_cells = int(df.isna().sum().sum())
    total_cells = int(df.shape[0] * df.shape[1])

    quality_results.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Missing_Cells": missing_cells,
        "Missing_Percentage": (
            round(missing_cells / total_cells * 100, 2)
            if total_cells > 0 else 0
        )
    })

quality_df = pd.DataFrame(quality_results)

display(inventory_df)
display(quality_df)

# ============================================================
# 4. TARGET CONSISTENCY CHECK
# ============================================================

print("\n" + "=" * 70)
print("TARGET CONSISTENCY CHECK")
print("=" * 70)

target_checks = {
    "MAO-B": {
        "dataset": "activity_raw",
        "expected_id": "CHEMBL2039"
    },
    "Alpha-Synuclein": {
        "dataset": "activity_raw",
        "expected_id": "CHEMBL6152"
    }
}

for target_name, config in target_checks.items():

    dataset_name = config["dataset"]
    expected_id = config["expected_id"]

    df = loaded_data.get(dataset_name)

    if df is None:
        print(f"{target_name}: DATASET NOT LOADED")
        continue

    if "target_chembl_id" not in df.columns:
        print(f"{target_name}: target_chembl_id COLUMN NOT FOUND")
        continue

    target_values = df["target_chembl_id"].dropna().astype(str).unique()

    print(f"\n{target_name}")
    print("Expected target ID:", expected_id)
    print("Observed target IDs:", target_values)

    if len(target_values) == 1 and target_values[0] == expected_id:
        print("Consistency: PASS")
    else:
        print("Consistency: REVIEW REQUIRED")

print("\nINITIAL VALIDATION FINISHED")

DATASET INVENTORY

[Compound_Jamur]
pubchem_compounds: NOT FOUND
validated_compounds: NOT FOUND

[MAO-B]
target_metadata: NOT FOUND
activity_raw: NOT FOUND

[Alpha-Synuclein]
target_metadata: FOUND | 1 rows × 9 columns
activity_raw: FOUND | 12,221 rows × 47 columns
assay_metadata: FOUND | 429 rows × 29 columns

BASIC DATA QUALITY CHECK


,Category,Dataset,File_Exists,Rows,Columns,File_Size_MB
0,Compound_Jamur,pubchem_compounds,False,NaN,NaN,NaN
1,Compound_Jamur,validated_compounds,False,NaN,NaN,NaN
2,MAO-B,target_metadata,False,NaN,NaN,NaN
3,MAO-B,activity_raw,False,NaN,NaN,NaN
4,Alpha-Synuclein,target_metadata,True,1.0,9.0,0.000
5,Alpha-Synuclein,activity_raw,True,12221.0,47.0,6.163
6,Alpha-Synuclein,assay_metadata,True,429.0,29.0,0.170


,Dataset,Rows,Columns,Duplicate_Rows,Missing_Cells,Missing_Percentage
0,target_metadata,1,9,0,0,0.00
1,activity_raw,12221,47,0,214382,37.32
2,assay_metadata,429,29,0,4688,37.68



TARGET CONSISTENCY CHECK

MAO-B
Expected target ID: CHEMBL2039
Observed target IDs: <StringArray>
['CHEMBL6152']
Length: 1, dtype: str
Consistency: REVIEW REQUIRED

Alpha-Synuclein
Expected target ID: CHEMBL6152
Observed target IDs: <StringArray>
['CHEMBL6152']
Length: 1, dtype: str
Consistency: PASS

INITIAL VALIDATION FINISHED


In [2]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/PROJECT SAINS DATA")
DATA_DIR = BASE_DIR / "DATA"

print("BASE_DIR exists:", BASE_DIR.exists())
print("DATA_DIR exists:", DATA_DIR.exists())

print("\nSTRUKTUR FOLDER PROJECT")
print("=" * 60)

if BASE_DIR.exists():
    for path in sorted(BASE_DIR.rglob("*")):
        if path.is_file():
            print("[FILE]", path.relative_to(BASE_DIR))
        elif path.is_dir():
            print("[DIR] ", path.relative_to(BASE_DIR))
else:
    print("BASE_DIR tidak ditemukan. Periksa lokasi Google Drive.")

BASE_DIR exists: True
DATA_DIR exists: True

STRUKTUR FOLDER PROJECT
[DIR]  DATA
[DIR]  DATA\Alpha-Synuclein
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_activity_raw.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_assay_distribution.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_assay_metadata_raw.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_bao_distribution.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_collection_log.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_collection_summary.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_endpoint_distribution.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_missing_summary.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_relation_distribution.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_target_metadata.csv
[FILE] DATA\Alpha-Synuclein\alpha_synuclein_unit_distribution.csv


In [15]:
# ============================================================
# 02.1 — ALPHA-SYNUCLEIN DATASET INVENTORY
# ============================================================

import pandas as pd
from pathlib import Path
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATH
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/PROJECT SAINS DATA")
DATA_DIR = BASE_DIR / "DATA"
ALPHA_DIR = DATA_DIR / "Alpha-Synuclein"

assert ALPHA_DIR.exists(), f"Folder tidak ditemukan: {ALPHA_DIR}"

# ------------------------------------------------------------
# 2. LOAD DATASETS
# ------------------------------------------------------------

FILE_PATHS = {
    "target_metadata": ALPHA_DIR / "alpha_synuclein_target_metadata.csv",
    "activity_raw": ALPHA_DIR / "alpha_synuclein_activity_raw.csv",
    "assay_metadata": ALPHA_DIR / "alpha_synuclein_assay_metadata_raw.csv"
}

loaded_data = {}
inventory_results = []

for name, path in FILE_PATHS.items():

    if path.exists():

        data = pd.read_csv(path, low_memory=False)
        loaded_data[name] = data

        inventory_results.append({
            "Dataset": name,
            "Status": "FOUND",
            "Rows": len(data),
            "Columns": len(data.columns),
            "Size_MB": round(path.stat().st_size / 1024**2, 3)
        })

    else:

        inventory_results.append({
            "Dataset": name,
            "Status": "NOT FOUND",
            "Rows": 0,
            "Columns": 0,
            "Size_MB": 0
        })

inventory_df = pd.DataFrame(inventory_results)

display(inventory_df)

# ------------------------------------------------------------
# 3. ASSIGN DATAFRAMES
# ------------------------------------------------------------

df_target = loaded_data.get("target_metadata")
df_alpha = loaded_data.get("activity_raw")
df_assay = loaded_data.get("assay_metadata")

# ------------------------------------------------------------
# 4. BASIC DATASET CHECK
# ------------------------------------------------------------

assert df_alpha is not None, "Activity dataset tidak ditemukan."

print("\nACTIVITY DATASET")
print("Shape:", df_alpha.shape)
print("Unique molecules:", df_alpha["molecule_chembl_id"].nunique())
print("Unique assays:", df_alpha["assay_chembl_id"].nunique())

if df_assay is not None:
    print("\nASSAY METADATA")
    print("Shape:", df_assay.shape)

print("\nDATASET INVENTORY COMPLETED")

,Dataset,Status,Rows,Columns,Size_MB
0,target_metadata,FOUND,1,9,0.000
1,activity_raw,FOUND,12221,47,6.163
2,assay_metadata,FOUND,429,29,0.170



ACTIVITY DATASET
Shape: (12221, 47)
Unique molecules: 11054
Unique assays: 429

ASSAY METADATA
Shape: (429, 29)

DATASET INVENTORY COMPLETED


In [16]:
# ============================================================
# 02.2 — BASIC DATA QUALITY AUDIT
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN BASIC DATA QUALITY AUDIT")
print("=" * 65)

# 1. BASIC INFORMATION
print("\n[1] DATASET INFORMATION")

print("Total records :", len(df_alpha))
print("Total columns :", len(df_alpha.columns))
print(
    "Memory usage  :",
    round(df_alpha.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

# 2. MISSING VALUES
print("\n[2] MISSING VALUE ANALYSIS")

missing_summary = pd.DataFrame({
    "Missing_Count": df_alpha.isna().sum(),
    "Missing_Percentage": (
        df_alpha.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
)

display(missing_summary.head(20))

# 3. DATA TYPES
print("\n[3] DATA TYPE CHECK")

dtype_summary = (
    df_alpha.dtypes.astype(str)
    .value_counts()
    .rename_axis("Data_Type")
    .reset_index(name="Column_Count")
)

display(dtype_summary)

# 4. DUPLICATE CHECK
print("\n[4] DUPLICATE CHECK")

print(
    "Exact duplicate rows:",
    df_alpha.duplicated().sum()
)

print(
    "Duplicate activity_id:",
    df_alpha["activity_id"].duplicated().sum()
)

print(
    "Missing activity_id:",
    df_alpha["activity_id"].isna().sum()
)

# 5. IMPORTANT COLUMN CHECK
print("\n[5] IMPORTANT COLUMN CHECK")

important_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "target_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value"
]

column_check = pd.DataFrame({
    "Column": important_columns,
    "Available": [
        col in df_alpha.columns
        for col in important_columns
    ]
})

display(column_check)

# 6. FINAL SUMMARY
print("\n" + "=" * 65)
print("BASIC DATA QUALITY AUDIT COMPLETED")
print("=" * 65)

print("Dataset shape:", df_alpha.shape)
print("Exact duplicates:", df_alpha.duplicated().sum())
print("Dataset remains unchanged.")

ALPHA-SYNUCLEIN BASIC DATA QUALITY AUDIT

[1] DATASET INFORMATION
Total records : 12221
Total columns : 47
Memory usage  : 23.34 MB

[2] MISSING VALUE ANALYSIS


,Missing_Count,Missing_Percentage
toid,12221,100.00
upper_value,12221,100.00
modality,12221,100.00
standard_upper_value,12221,100.00
data_validity_comment,12209,99.90
data_validity_description,12209,99.90
assay_variant_accession,12206,99.88
assay_variant_mutation,12206,99.88
molecule_pref_name,11853,96.99
standard_text_value,11748,96.13



[3] DATA TYPE CHECK


,Data_Type,Column_Count
0,str,33
1,float64,8
2,int64,6



[4] DUPLICATE CHECK
Exact duplicate rows: 0
Duplicate activity_id: 0
Missing activity_id: 0

[5] IMPORTANT COLUMN CHECK


,Column,Available
0,molecule_chembl_id,True
1,assay_chembl_id,True
2,target_chembl_id,True
3,standard_type,True
4,standard_value,True
5,standard_units,True
6,standard_relation,True
7,pchembl_value,True



BASIC DATA QUALITY AUDIT COMPLETED
Dataset shape: (12221, 47)
Exact duplicates: 0
Dataset remains unchanged.


In [17]:
# ============================================================
# 02.3 — TARGET, ENDPOINT & ASSAY VALIDATION
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN TARGET & ENDPOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. TARGET IDENTITY VALIDATION
# ------------------------------------------------------------

print("\n[1] TARGET IDENTITY")

target_columns = [
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "target_tax_id"
]

target_summary = []

for col in target_columns:
    if col in df_alpha.columns:
        values = df_alpha[col].value_counts(dropna=False)

        for value, count in values.items():
            target_summary.append({
                "Column": col,
                "Value": value,
                "Count": count
            })

target_summary = pd.DataFrame(target_summary)
display(target_summary)

# Validasi target utama
expected_target = "CHEMBL6152"

observed_targets = set(
    df_alpha["target_chembl_id"].dropna()
)

if observed_targets == {expected_target}:
    print("\nTarget ID: VALID")
else:
    print("\nWARNING: Target ID tidak konsisten!")
    print("Observed:", observed_targets)


# ------------------------------------------------------------
# 2. ENDPOINT & ACTIVITY MEASUREMENT
# ------------------------------------------------------------

print("\n[2] ENDPOINT & ACTIVITY MEASUREMENT")

endpoint_columns = [
    "standard_type",
    "standard_units",
    "standard_relation"
]

for col in endpoint_columns:
    if col in df_alpha.columns:
        print(f"\n{col.upper()}")
        display(
            df_alpha[col]
            .value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="Count")
        )


# ------------------------------------------------------------
# 3. ASSAY & BAO CHARACTERISTICS
# ------------------------------------------------------------

print("\n[3] ASSAY & BAO CHARACTERISTICS")

assay_columns = [
    "assay_type",
    "bao_endpoint",
    "bao_format"
]

for col in assay_columns:
    if col in df_alpha.columns:
        print(f"\n{col.upper()}")
        display(
            df_alpha[col]
            .value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="Count")
        )


# ------------------------------------------------------------
# 4. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TARGET & ENDPOINT VALIDATION COMPLETED")
print("=" * 65)

print("Target ID       :", observed_targets)
print("Total records   :", len(df_alpha))
print("Unique endpoints:",
      df_alpha["standard_type"].nunique())
print("Unique assays   :",
      df_alpha["assay_chembl_id"].nunique())

print("\nDataset remains unchanged.")

ALPHA-SYNUCLEIN TARGET & ENDPOINT VALIDATION

[1] TARGET IDENTITY


,Column,Value,Count
0,target_chembl_id,CHEMBL6152,12221
1,target_pref_name,Alpha-synuclein,12221
2,target_organism,Homo sapiens,12221
3,target_tax_id,9606,12221



Target ID: VALID

[2] ENDPOINT & ACTIVITY MEASUREMENT

STANDARD_TYPE


,standard_type,Count
0,Potency,10056
1,Inhibition,770
2,Activity,510
3,IC50,192
4,Ki,145
5,Kd,141
6,EC50,92
7,kon,84
8,k_off,84
9,Flu intensity,59



STANDARD_UNITS


,standard_units,Count
0,nM,10593
1,%,1059
2,NaN,364
3,s-1,84
4,RU,47
5,hr,41
6,10^-3/hr,9
7,a.u.,8
8,/hr,7
9,nm,6


... (5 outputs left out)

,bao_endpoint,Count
0,BAO_0000186,10056
1,BAO_0000201,770
2,BAO_0001103,230
3,BAO_0000375,219
4,BAO_0000179,208
5,BAO_0000190,192
6,BAO_0000192,145
7,BAO_0000034,141
8,BAO_0000188,92
9,BAO_0000480,84



BAO_FORMAT


,bao_format,Count
0,BAO_0000019,10310
1,BAO_0000357,1428
2,BAO_0000219,475
3,BAO_0000221,8



TARGET & ENDPOINT VALIDATION COMPLETED
Target ID       : {'CHEMBL6152'}
Total records   : 12221
Unique endpoints: 19
Unique assays   : 429

Dataset remains unchanged.


In [18]:
# ============================================================
# 02.4 — POTENCY INVESTIGATION
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN POTENCY INVESTIGATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. FILTER POTENCY
# ------------------------------------------------------------

df_potency = df_alpha[
    df_alpha["standard_type"].eq("Potency")
].copy()

print("\n[1] DATASET OVERVIEW")
print("Total Potency records :", len(df_potency))
print("Unique compounds      :",
      df_potency["molecule_chembl_id"].nunique())
print("Unique assays         :",
      df_potency["assay_chembl_id"].nunique())


# ------------------------------------------------------------
# 2. ASSAY DISTRIBUTION
# ------------------------------------------------------------

print("\n[2] TOP 20 ASSAYS")

assay_summary = (
    df_potency.groupby(
        ["assay_chembl_id", "assay_description"],
        dropna=False
    )
    .size()
    .reset_index(name="Record_Count")
    .sort_values("Record_Count", ascending=False)
)

display(assay_summary.head(20))


# ------------------------------------------------------------
# 3. ENDPOINT AND ASSAY TYPE
# ------------------------------------------------------------

print("\n[3] ENDPOINT DISTRIBUTION")

display(
    df_potency["bao_endpoint"]
    .value_counts(dropna=False)
    .rename_axis("BAO_Endpoint")
    .reset_index(name="Count")
)

print("\nASSAY TYPE DISTRIBUTION")

display(
    df_potency["assay_type"]
    .value_counts(dropna=False)
    .rename_axis("Assay_Type")
    .reset_index(name="Count")
)


# ------------------------------------------------------------
# 4. UNIT AND RELATION DISTRIBUTION
# ------------------------------------------------------------

print("\n[4] STANDARD UNITS")

display(
    df_potency["standard_units"]
    .value_counts(dropna=False)
)

print("\nSTANDARD RELATION")

display(
    df_potency["standard_relation"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. ACTIVITY VALUE ANALYSIS
# ------------------------------------------------------------

print("\n[5] STANDARD VALUE ANALYSIS")

potency_values = pd.to_numeric(
    df_potency["standard_value"],
    errors="coerce"
)

print(potency_values.describe())

print("\nMissing values :", potency_values.isna().sum())
print("Zero values    :", (potency_values == 0).sum())
print("Negative values:", (potency_values < 0).sum())


# ------------------------------------------------------------
# 6. PCHEMBL AVAILABILITY
# ------------------------------------------------------------

print("\n[6] PCHEMBL AVAILABILITY")

pchembl_available = df_potency["pchembl_value"].notna().sum()
pchembl_missing = df_potency["pchembl_value"].isna().sum()

print("Available:", pchembl_available)
print("Missing  :", pchembl_missing)

print(
    "Availability (%):",
    round(pchembl_available / len(df_potency) * 100, 2)
)


# ------------------------------------------------------------
# 7. SAMPLE RECORDS
# ------------------------------------------------------------

print("\n[7] POTENCY RECORD SAMPLE")

sample_columns = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "assay_type",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "bao_endpoint"
]

display(
    df_potency[sample_columns].sample(
        min(15, len(df_potency)),
        random_state=42
    )
)


# ------------------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("POTENCY INVESTIGATION COMPLETED")
print("=" * 65)

print("Total records      :", len(df_potency))
print("Unique assays      :", df_potency["assay_chembl_id"].nunique())
print("Available pChEMBL  :", pchembl_available)
print("Missing pChEMBL    :", pchembl_missing)

print("\nOriginal dataset remains unchanged.")

ALPHA-SYNUCLEIN POTENCY INVESTIGATION

[1] DATASET OVERVIEW
Total Potency records : 10056
Unique compounds      : 9989
Unique assays         : 2

[2] TOP 20 ASSAYS


,assay_chembl_id,assay_description,Record_Count
1,CHEMBL2354282,PubChem BioAssay. qHTS of alpha-syn Inhibitors...,10019
0,CHEMBL1613965,PUBCHEM_BIOASSAY: Alpha-synuclein ThS binding ...,37



[3] ENDPOINT DISTRIBUTION


,BAO_Endpoint,Count
0,BAO_0000186,10056



ASSAY TYPE DISTRIBUTION


,Assay_Type,Count
0,F,10019
1,B,37


... (3 outputs left out)

standard_relation
NaN    10019
=         37
Name: count, dtype: int64


[5] STANDARD VALUE ANALYSIS
count    10056.000000
mean     19256.984559
std      11906.290760
min         20.000000
25%       7943.300000
50%      17782.800000
75%      31622.800000
max      56234.100000
Name: standard_value, dtype: float64

Missing values : 0
Zero values    : 0
Negative values: 0

[6] PCHEMBL AVAILABILITY
Available: 37
Missing  : 10019
Availability (%): 0.37

[7] POTENCY RECORD SAMPLE


,activity_id,molecule_chembl_id,assay_chembl_id,assay_type,standard_type,standard_relation,standard_value,standard_units,pchembl_value,bao_endpoint
8336,13184189,CHEMBL1577485,CHEMBL2354282,F,Potency,NaN,15848.9,nM,NaN,BAO_0000186
5326,13162304,CHEMBL1709829,CHEMBL2354282,F,Potency,NaN,25118.9,nM,NaN,BAO_0000186
1228,13140144,CHEMBL1500321,CHEMBL2354282,F,Potency,NaN,10000.0,nM,NaN,BAO_0000186
8700,13184553,CHEMBL2135932,CHEMBL2354282,F,Potency,NaN,28183.8,nM,NaN,BAO_0000186
186,13136104,CHEMBL1699228,CHEMBL2354282,F,Potency,NaN,35481.3,nM,NaN,BAO_0000186
9385,13188244,CHEMBL1587179,CHEMBL2354282,F,Potency,NaN,8912.5,nM,NaN,BAO_0000186
8806,13187665,CHEMBL2138177,CHEMBL2354282,F,Potency,NaN,5623.4,nM,NaN,BAO_0000186
8666,13184519,CHEMBL2143481,CHEMBL2354282,F,Potency,NaN,22387.2,nM,NaN,BAO_0000186
3061,13150992,CHEMBL1701081,CHEMBL2354282,F,Potency,NaN,12589.3,nM,NaN,BAO_0000186
1296,13140212,CHEMBL1733422,CHEMBL2354282,F,Potency,NaN,11220.2,nM,NaN,BAO_0000186



POTENCY INVESTIGATION COMPLETED
Total records      : 10056
Unique assays      : 2
Available pChEMBL  : 37
Missing pChEMBL    : 10019

Original dataset remains unchanged.


In [19]:
# ============================================================
# 02.5 — FINAL DUPLICATE VALIDATION
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN FINAL DUPLICATE VALIDATION")
print("=" * 65)

df_check = df_alpha.copy()

# ------------------------------------------------------------
# 1. RECORD ID AUDIT
# ------------------------------------------------------------

print("\n[1] RECORD ID AUDIT")

record_duplicate = df_check["record_id"].duplicated(keep=False)

print("Missing record_id       :", df_check["record_id"].isna().sum())
print("Duplicate record rows   :", record_duplicate.sum())
print("Unique repeated IDs     :",
      df_check.loc[record_duplicate, "record_id"].nunique())


# ------------------------------------------------------------
# 2. ACTIVITY ID AUDIT
# ------------------------------------------------------------

print("\n[2] ACTIVITY ID AUDIT")

print("Missing activity_id     :", df_check["activity_id"].isna().sum())
print("Duplicate activity_id   :",
      df_check["activity_id"].duplicated().sum())
print("Unique activity_id      :",
      df_check["activity_id"].nunique())


# ------------------------------------------------------------
# 3. EXACT DUPLICATE AUDIT
# ------------------------------------------------------------

print("\n[3] EXACT DUPLICATE AUDIT")

print("Exact duplicate rows    :", df_check.duplicated().sum())


# ------------------------------------------------------------
# 4. MEASUREMENT SIGNATURE AUDIT
# ------------------------------------------------------------

print("\n[4] MEASUREMENT SIGNATURE AUDIT")

measurement_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

measurement_duplicates = df_check[
    df_check.duplicated(
        subset=measurement_columns,
        keep=False
    )
].copy()

repeated_groups = measurement_duplicates.groupby(
    measurement_columns,
    dropna=False
).ngroups

print("Repeated measurement rows :", len(measurement_duplicates))
print("Repeated measurement groups:", repeated_groups)


# ------------------------------------------------------------
# 5. DUPLICATE RECORD SAMPLES
# ------------------------------------------------------------

print("\n[5] RECORD ID DUPLICATE SAMPLES")

if record_duplicate.any():
    display(
        df_check.loc[
            record_duplicate,
            ["activity_id", "record_id", "molecule_chembl_id",
             "assay_chembl_id", "standard_type",
             "standard_value", "standard_units"]
        ].sort_values("record_id").head(20)
    )
else:
    print("Tidak ditemukan record_id berulang.")


print("\n[6] MEASUREMENT DUPLICATE SAMPLES")

if not measurement_duplicates.empty:
    display(
        measurement_duplicates[
            ["activity_id", "record_id", *measurement_columns]
        ].sort_values(measurement_columns).head(20)
    )
else:
    print("Tidak ditemukan measurement signature berulang.")


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL DUPLICATE VALIDATION COMPLETED")
print("=" * 65)

print("Total records            :", len(df_check))
print("Duplicate record rows    :", record_duplicate.sum())
print("Duplicate activity_id    :",
      df_check["activity_id"].duplicated().sum())
print("Exact duplicate rows     :", df_check.duplicated().sum())
print("Repeated measurement rows:", len(measurement_duplicates))

print("\nDataset asli tidak diubah.")

ALPHA-SYNUCLEIN FINAL DUPLICATE VALIDATION

[1] RECORD ID AUDIT
Missing record_id       : 0
Duplicate record rows   : 1402
Unique repeated IDs     : 325

[2] ACTIVITY ID AUDIT
Missing activity_id     : 0
Duplicate activity_id   : 0
Unique activity_id      : 12221

[3] EXACT DUPLICATE AUDIT
Exact duplicate rows    : 0

[4] MEASUREMENT SIGNATURE AUDIT
Repeated measurement rows : 58
Repeated measurement groups: 29

[5] RECORD ID DUPLICATE SAMPLES


,activity_id,record_id,molecule_chembl_id,assay_chembl_id,standard_type,standard_value,standard_units
2,3396892,930755,CHEMBL1213645,CHEMBL1218779,Inhibition,NaN,%
4,3396898,930755,CHEMBL1213645,CHEMBL1218780,Activity,NaN,NaN
7,3396901,930755,CHEMBL1213645,CHEMBL1218781,Inhibition,26.0,%
3,3396893,930756,CHEMBL1213646,CHEMBL1218779,Inhibition,NaN,%
5,3396899,930756,CHEMBL1213646,CHEMBL1218780,Activity,NaN,NaN
6,3396900,930756,CHEMBL1213646,CHEMBL1218781,Inhibition,24.0,%
30,3446915,953293,CHEMBL1242187,CHEMBL1250268,Activity,50.0,%
34,3448364,953293,CHEMBL1242187,CHEMBL1250273,Activity,NaN,NaN
31,3446916,953293,CHEMBL1242187,CHEMBL1250270,Activity,70.0,%
25,3446910,953293,CHEMBL1242187,CHEMBL1250262,Activity,410.0,a.u.



[6] MEASUREMENT DUPLICATE SAMPLES


,activity_id,record_id,molecule_chembl_id,assay_chembl_id,standard_type,standard_relation,standard_value,standard_units
1461,13140377,1254429,CHEMBL1316480,CHEMBL2354282,Potency,NaN,35481.3,nM
3161,13151092,1711141,CHEMBL1316480,CHEMBL2354282,Potency,NaN,35481.3,nM
4000,13154946,1142064,CHEMBL1448592,CHEMBL2354282,Potency,NaN,10000.0,nM
7079,13173303,1115328,CHEMBL1448592,CHEMBL2354282,Potency,NaN,10000.0,nM
6133,13166124,1856597,CHEMBL157101,CHEMBL2354282,Potency,NaN,3162.3,nM
6386,13169384,1263470,CHEMBL157101,CHEMBL2354282,Potency,NaN,3162.3,nM
11876,28237931,4473787,CHEMBL1908919,CHEMBL5736378,Kd,=,24.3,nM
11909,28237997,4473787,CHEMBL1908919,CHEMBL5736378,Kd,=,24.3,nM
11878,28237933,4473787,CHEMBL1908919,CHEMBL5736378,k_off,=,NaN,s-1
11911,28237999,4473787,CHEMBL1908919,CHEMBL5736378,k_off,=,NaN,s-1



FINAL DUPLICATE VALIDATION COMPLETED
Total records            : 12221
Duplicate record rows    : 1402
Duplicate activity_id    : 0
Exact duplicate rows     : 0
Repeated measurement rows: 58

Dataset asli tidak diubah.


In [20]:
# ============================================================
# 02.6 — VALIDATION CONCLUSION
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN DATA VALIDATION CONCLUSION")
print("=" * 65)

# 1. Dataset overview
print("\n[1] DATASET OVERVIEW")
print("Target              : Alpha-synuclein")
print("Target ChEMBL ID    : CHEMBL6152")
print("Organism            : Homo sapiens")
print("Total records       :", len(df_alpha))
print("Total columns       :", len(df_alpha.columns))

# 2. Data quality
print("\n[2] DATA QUALITY")
print("Missing activity_id :", df_alpha["activity_id"].isna().sum())
print("Exact duplicates    :", df_alpha.duplicated().sum())

# 3. Endpoint overview
print("\n[3] ENDPOINT OVERVIEW")
print("Unique standard types:",
      df_alpha["standard_type"].nunique())

print("Unique assays:",
      df_alpha["assay_chembl_id"].nunique())

# 4. Potency overview
print("\n[4] POTENCY OVERVIEW")
print("Potency records     :", len(df_potency))
print("Potency units       :",
      df_potency["standard_units"].nunique())

print("Available pChEMBL   :",
      df_potency["pchembl_value"].notna().sum())

# 5. Duplicate overview
print("\n[5] DUPLICATE OVERVIEW")
print("Repeated record rows:",
      record_duplicate.sum())

print("Repeated measurement rows:",
      len(measurement_duplicates))

# 6. Validation decision
print("\n[6] VALIDATION DECISION")

print("- Dataset berhasil melewati pemeriksaan dasar.")
print("- Target dan identitas organisme telah tervalidasi.")
print("- Endpoint dan satuan pengukuran bervariasi.")
print("- Potency merupakan endpoint dominan.")
print("- Pengulangan record_id tidak otomatis dianggap duplikasi.")
print("- Kandidat pengukuran berulang perlu ditinjau lebih lanjut.")
print("- Tidak ada record yang dihapus pada tahap validasi.")

print("\n" + "=" * 65)
print("NOTEBOOK 02 — DATA VALIDATION COMPLETED")
print("=" * 65)

ALPHA-SYNUCLEIN DATA VALIDATION CONCLUSION

[1] DATASET OVERVIEW
Target              : Alpha-synuclein
Target ChEMBL ID    : CHEMBL6152
Organism            : Homo sapiens
Total records       : 12221
Total columns       : 47

[2] DATA QUALITY
Missing activity_id : 0
Exact duplicates    : 0

[3] ENDPOINT OVERVIEW
Unique standard types: 19
Unique assays: 429

[4] POTENCY OVERVIEW
Potency records     : 10056
Potency units       : 1
Available pChEMBL   : 37

[5] DUPLICATE OVERVIEW
Repeated record rows: 1402
Repeated measurement rows: 58

[6] VALIDATION DECISION
- Dataset berhasil melewati pemeriksaan dasar.
- Target dan identitas organisme telah tervalidasi.
- Endpoint dan satuan pengukuran bervariasi.
- Potency merupakan endpoint dominan.
- Pengulangan record_id tidak otomatis dianggap duplikasi.
- Kandidat pengukuran berulang perlu ditinjau lebih lanjut.
- Tidak ada record yang dihapus pada tahap validasi.

NOTEBOOK 02 — DATA VALIDATION COMPLETED
